In [1]:
import pandas as pd
import google as genai
import google.generativeai as genai
# Similaridade de cosseno
from sklearn.metrics.pairwise import cosine_similarity

import warnings 
warnings.filterwarnings('ignore')

C:\Users\gusta\AppData\Local\Temp\ipykernel_10388\599227194.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
# API Key
GOOGLE_API_KEY = ""

genai.configure(api_key=GOOGLE_API_KEY)

# Construir Chatbot de RH com RAG

Problema de Negócio: A equipe de RH quer um chatbot que responda perguntas de funcionários com base exclusivamente nos documentos de políticas da empresa, sem inventar respostas.

In [29]:
documentos_rh = [
  {"titulo":"Política de Férias","conteudo":"Todo funcionário com mais de 12 meses de contrato tem direito a 30 dias de férias remuneradas. A solicitação deve ser feita com 60 dias de antecedência. É permitido vender até 10 dias de férias."},
  {"titulo": "Política de Home Office", "conteudo": "O modelo de trabalho padrão é híbrido. Para os dias de home office, a empresa oferece uma ajuda de custo de R$ 100,00 mensais para despesas com internet e energia"},
  {"titulo": "Política de Licença Paternidade", "conteudo": "A empresa oferece uma licença paternidade estendida de 30 dias corridos para novos pais, superior aos 5 dias previstos em lei. A licença deve começar na data de nascimento do bebê."}
]

df_docs = pd.DataFrame(documentos_rh)

In [30]:
df_docs

,titulo,conteudo
0,Política de Férias,Todo funcionário com mais de 12 meses de contr...
1,Política de Home Office,O modelo de trabalho padrão é híbrido. Para os...
2,Política de Licença Paternidade,A empresa oferece uma licença paternidade este...


In [31]:
model_chat = genai.GenerativeModel('gemini-2.5-flash')

# Testar o LLM puro
pergunta_usuario = "Quantos dias de licença paternindade a empresa oferece?"

response_sem_rag = model_chat.generate_content(pergunta_usuario)

In [32]:
print(f"Pergunta: {pergunta_usuario}")
print(f"Resposta do LLM puro: {response_sem_rag.text}")

Pergunta: Quantos dias de licença paternindade a empresa oferece?
Resposta do LLM puro: Para saber exatamente quantos dias de licença paternidade **a sua empresa** oferece, você precisa verificar as políticas internas dela.

No Brasil, a regra geral é a seguinte:

1.  **Mínimo Legal:** O período mínimo de licença-paternidade é de **5 dias corridos**, conforme a Constituição Federal e a CLT (Consolidação das Leis do Trabalho).

2.  **Programa Empresa Cidadã:** Empresas que aderem voluntariamente ao **Programa Empresa Cidadã** podem estender esse período para **20 dias**. Essa extensão é um benefício fiscal para a empresa e exige que o pai solicite a prorrogação em até 2 dias úteis após o parto e comprove a participação em programa ou atividade de orientação sobre paternidade responsável.

**Como você pode descobrir o período exato para a sua empresa:**

*   **Consulte o RH (Recursos Humanos):** É a forma mais direta e segura de obter essa informação.
*   **Verifique o seu contrato de tr

In [ ]:
# Resposta com RAG

# 1. Retrieval - Criar embeddings (representações vetoriais da pergunta) e encontrar o documento relevante 
# Embeddings -> Representações vatoriais da pargunda. A IA só entende números e o embedding é um vetor de números que carregam o significado da palavra
# Em resumo, o embedding cria um vetor númerico (como se fosse uma seta no plano carteziano) e a similaridade entre um ponto e outro é fetia utilizando a distancia alguar entre os pontos (vetores) e por isso é feito utilizando a similaridade por cosseno

# Aqui será utilizado modelos feitos para fazer a conversão de texto em vetores númericos que carregam o significado da frase
embedding_pergunta = genai.embed_content(
  model="models/gemini-embedding-001", 
  content=pergunta_usuario,
)['embedding']

# Embedding das respostas da base de conhecimento
df_docs['Embedding'] = df_docs['conteudo'].apply(lambda x: genai.embed_content(model="models/gemini-embedding-001", content=x)['embedding'])

# Similaridade entre a pergunta e cada uma das respostas (conteúdo). Ira retornar um vetor que vai pegar o valor que ta no topo, primeira linha e coluna
df_docs['Similaridade'] = df_docs['Embedding'].apply(lambda x: cosine_similarity([x], [embedding_pergunta])[0][0])
# Quanto mais próximo de 1 a similaridade tiver, maior é a chance do treixo de conteudo trazer a resposta compatível com o que foi perguntado

In [34]:
df_docs

,titulo,conteudo,Embedding,Similaridade
0,Política de Férias,Todo funcionário com mais de 12 meses de contr...,"[-0.015275286, 0.026008159, 0.008141993, -0.06...",0.712756
1,Política de Home Office,O modelo de trabalho padrão é híbrido. Para os...,"[0.0037565548, 0.018948268, 0.008957084, -0.08...",0.652176
2,Política de Licença Paternidade,A empresa oferece uma licença paternidade este...,"[-0.014370553, 0.02875204, 0.011555173, -0.052...",0.883246


In [38]:
# 2. Generation - Montar o prompt aumentado e gerar a resposta

# O documento relevante é a tecnica para pegar a informação com maior similaridade - iloc serve para fazer uma busca por indice e sort_values ordena o dataframe pela coluna de similaridade
documento_relevante = df_docs.sort_values(by='Similaridade', ascending=False).iloc[0]
prompt_com_rag = f"""
Com base APENAS no contexto abaixo, responda à pergunta do usuário. Se a resposta não estiver no contexto, diga "Não encontrei essa informação nos documentos".

**Contexto:**
"{documento_relevante['conteudo']}"

**Pergunta:**
"{pergunta_usuario}"
"""

In [40]:
print(prompt_com_rag)


Com base APENAS no contexto abaixo, responda à pergunta do usuário. Se a resposta não estiver no contexto, diga "Não encontrei essa informação nos documentos".

**Contexto:**
"A empresa oferece uma licença paternidade estendida de 30 dias corridos para novos pais, superior aos 5 dias previstos em lei. A licença deve começar na data de nascimento do bebê."

**Pergunta:**
"Quantos dias de licença paternindade a empresa oferece?"



In [41]:
# Gera resposta utilizando o prompt aumentado
response_com_rag = model_chat.generate_content(prompt_com_rag)
print(response_com_rag.text)

A empresa oferece 30 dias corridos de licença paternidade.


In [42]:
# Função para código reutilizável

def chatbot_rh(pergunta):
  # 1. Retrieval
  embedding_pergunta = genai.embed_content(
    model="models/gemini-embedding-001", 
    content=pergunta,
  )['embedding']

  df_docs['Similaridade'] = df_docs['Embedding'].apply(lambda x: cosine_similarity([x], [embedding_pergunta])[0][0])
  
  documento_relevante = df_docs.sort_values(by='Similaridade', ascending=False).iloc[0]
  
  # 2. Generation
  prompt_final = f"""
  Com base APENAS no contexto abaixo, responda à pergunta do usuário. Se a resposta não estiver no contexto, diga "Não encontrei essa informação nos documentos".

  **Contexto:**
  "{documento_relevante['conteudo']}"

  **Pergunta:**
  "{pergunta}"
  """
  
  response = model_chat.generate_content(prompt_final)
  
  return response.text

In [43]:
nova_pergunta = "A empresa oferece ajuda de custo para home office?"
print(f"Pergunta: {nova_pergunta}")
print(f"Resposta: {chatbot_rh(nova_pergunta)}")

Pergunta: A empresa oferece ajuda de custo para home office?
Resposta: Sim, a empresa oferece uma ajuda de custo de R$ 100,00 mensais para despesas com internet e energia nos dias de home office.
